In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# import os

# zip_file_path = "/content/drive/My Drive/Imagenet-100.zip"
# if os.path.exists(zip_file_path):
#     print("Zip file exists!")
# else:
#     print("Zip file does not exist!")

In [ ]:
import os
import shutil
import math
import random
import time
import datetime
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from PIL import Image, ImageFilter, ImageOps

import wandb 
from tqdm import tqdm 

def setup_dataset():
    train_zip = "/content/drive/MyDrive/ImageNet-1k-0.zip"
    val_zip = "/content/drive/MyDrive/ImageNet-1k-valid.zip"

    train_unzip_dir = "/content/ImageNet-1k-0"
    val_unzip_dir = "/content/ImageNet-1k-valid"

    # Create destination directories if they don't exist
    os.makedirs(train_unzip_dir, exist_ok=True)
    os.makedirs(val_unzip_dir, exist_ok=True)

    print("UNZIPPING TRAINING DATA!")
    # Check if the training directory is empty
    if not os.listdir(train_unzip_dir):
        !unzip -q "{train_zip}" -d "{train_unzip_dir}"
    else:
        print("Training data already unzipped!")

    print("UNZIPPING VALIDATION DATA!")
    if not os.listdir(val_unzip_dir):
        !unzip -q "{val_zip}" -d "{val_unzip_dir}"
    else:
        print("Validation data already unzipped!")

    return train_unzip_dir, val_unzip_dir


In [ ]:
train_dir, val_dir = setup_dataset()

UNZIPPING TRAINING DATA!
Training data already unzipped!
UNZIPPING VALIDATION DATA!
Validation data already unzipped!


In [ ]:
################################################################################
# Step 2: 3Augment (from DeiT III)
################################################################################

class GaussianBlur(object):
    """Apply Gaussian Blur to the PIL image."""
    def __init__(self, p=0.1, radius_min=0.1, radius_max=2.):
        self.prob = p
        self.radius_min = radius_min
        self.radius_max = radius_max

    def __call__(self, img):
        do_it = (random.random() <= self.prob)
        if not do_it:
            return img
        return img.filter(
            ImageFilter.GaussianBlur(
                radius=random.uniform(self.radius_min, self.radius_max)
            )
        )

class Solarization(object):
    """Apply Solarization to the PIL image."""
    def __init__(self, p=0.2):
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            return ImageOps.solarize(img)
        return img

class GrayScale(object):
    """Random grayscale."""
    def __init__(self, p=0.2):
        self.p = p
        self.transf = transforms.Grayscale(3)

    def __call__(self, img):
        if random.random() < self.p:
            return self.transf(img)
        return img

def three_augment_transform(
    input_size=224,
    color_jitter=0.3,
    interpolation=3,  # 'bicubic'
):
    """
    3Augment: random resized crop + random horizontal flip
              + random choice among [GrayScale, Solarization, GaussianBlur]
              + optional color jitter
              + normalization
    """
    primary_tfl = [
        transforms.RandomResizedCrop(input_size, scale=(0.08,1.0), interpolation=interpolation),
        transforms.RandomHorizontalFlip(),
    ]
    # exactly one from the triple
    secondary_tfl = transforms.RandomChoice([
        GrayScale(p=1.0),
        Solarization(p=1.0),
        GaussianBlur(p=1.0),
    ])
    final_tfl = [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ]
    augment_list = primary_tfl + [secondary_tfl]
    if color_jitter > 0.0:
        augment_list.append(transforms.ColorJitter(color_jitter, color_jitter, color_jitter))
    augment_list += final_tfl
    return transforms.Compose(augment_list)

def eval_transform(input_size=224, interpolation=3):
    """
    Standard validation transform (same as typical DeiT/ViT):
    resize -> center crop -> normalize.
    """
    return transforms.Compose([
        transforms.Resize(int(input_size / 0.875), interpolation=interpolation),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

################################################################################
# Step 3: Model definition - DeiT-Tiny w/ LayerScale
################################################################################

class DropPath(nn.Module):
    """Drop paths (Stochastic Depth)."""
    def __init__(self, drop_prob=0.):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,)* (x.ndim-1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()  # binarize
        output = x.div(keep_prob) * random_tensor
        return output

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class Attention(nn.Module):
    """Standard ViT attention with num_heads=3 for DeiT-Tiny."""
    def __init__(self, dim, num_heads=3, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B,N,C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C//self.num_heads).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1,2).reshape(B,N,C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class LayerScaleBlock(nn.Module):
    """LayerScale block used by DeiT III for small variants (init=1e-4)."""
    def __init__(self, dim, num_heads=3, mlp_ratio=4., qkv_bias=True,
                 drop=0., attn_drop=0., drop_path=0., init_values=1e-4,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                              attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim*mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)

        self.gamma_1 = nn.Parameter(init_values*torch.ones((dim)), requires_grad=True)
        self.gamma_2 = nn.Parameter(init_values*torch.ones((dim)), requires_grad=True)

    def forward(self, x):
        x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x)))
        x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        return x

class PatchEmbed(nn.Module):
    """Standard patch embed for 224->14x14 patches if patch_size=16."""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size//patch_size)*(img_size//patch_size)
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)           # shape (B, embed_dim, H/patch, W/patch)
        x = x.flatten(2).transpose(1,2)  # (B, #patches, embed_dim)
        return x

In [ ]:
class DeiT_Tiny_LS(nn.Module):
    """
    A modified DeiT-Tiny with LayerScale blocks that supports:
      - Multiple CLS token extractions at arbitrary block indices.
      - Replacing the used CLS token with a NEW one for subsequent blocks.
      - Final head is a linear over the concatenation of all extracted CLS tokens.
      - By default, extraction_blocks = [depth-1], so it behaves similarly to the
        standard single-CLS approach (but each extracted CLS uses its own LN).
    """
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        num_classes=100,
        embed_dim=192,
        depth=12,
        num_heads=3,
        mlp_ratio=4.0,
        qkv_bias=True,
        drop_rate=0.,
        attn_drop_rate=0.,
        drop_path_rate=0.05,
        init_values=1e-4,
        use_cls_token=True,
        extraction_blocks=[3, 7, 11],  # list of block indices where we "tap out" & re-insert
    ):
        super().__init__()
        self.use_cls_token = use_cls_token
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.depth = depth

        if extraction_blocks is None:
            # default: single extraction at the last block
            extraction_blocks = [depth - 1]
        self.extraction_blocks = extraction_blocks
        self.num_extractions = len(extraction_blocks)

        # patch embedding
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size,
            in_chans=in_chans, embed_dim=embed_dim
        )
        num_patches = self.patch_embed.num_patches

        # pos embed
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))

        # (optional) original cls token
        if self.use_cls_token:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        else:
            self.cls_token = None

        # prepare the Transformer blocks
        dpr_vals = [drop_path_rate for _ in range(depth)]
        self.blocks = nn.ModuleList([
            LayerScaleBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr_vals[i],
                init_values=init_values,
            )
            for i in range(depth)
        ])

        self.exit_norms = nn.ModuleList([
            nn.LayerNorm(embed_dim) for _ in range(self.num_extractions)
        ])

        self.new_cls_tokens = nn.Parameter(torch.zeros(self.num_extractions, 1, embed_dim))


        self.head = nn.Linear(embed_dim * self.num_extractions, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        if self.cls_token is not None:
            nn.init.trunc_normal_(self.cls_token, std=0.02)

        nn.init.trunc_normal_(self.new_cls_tokens, std=0.02)

        # init all other submodules
        def _init(m):
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.bias, 0)
                nn.init.constant_(m.weight, 1.0)

        self.apply(_init)

    def forward_features(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)
        x = x + self.pos_embed  # (B, N, C)

        # Insert the original/first cls token
        if self.use_cls_token:
            # shape (B, 1, C)
            first_cls = self.cls_token.expand(B, -1, -1)
            x = torch.cat([first_cls, x], dim=1)  # (B, 1+N, C)

        # We'll collect each extracted CLS here
        extracted_cls_list = []

        insertion_idx = 0  # which extraction index we are on
        for i, blk in enumerate(self.blocks):
            x = blk(x)

            # if this block is in the user-specified extraction list:
            if i in self.extraction_blocks:
                # Extract the CLS at x[:,0]
                cls_i = x[:, 0]  # (B, C)
                # LayerNorm it with the corresponding LN
                cls_i = self.exit_norms[insertion_idx](cls_i)
                extracted_cls_list.append(cls_i)

                # If this is NOT the last extraction, replace x[:,0] with a new CLS
                if insertion_idx < self.num_extractions - 1:
                    new_cls = self.new_cls_tokens[insertion_idx + 1].expand(B, -1, -1)  # (B,1,C)
                    # re-inject
                    x = torch.cat([new_cls, x[:, 1:]], dim=1)

                insertion_idx += 1

        # Concat along dim=-1 => shape: (B, num_extractions*C)
        multi_cls = torch.cat(extracted_cls_list, dim=-1)
        return multi_cls

    def forward(self, x):
        multi_cls = self.forward_features(x)
        logits = self.head(multi_cls)
        return logits

In [ ]:
class MixupCutmixCollator:
    """
    Applies mixup and cutmix to a batch, with a 50/50 chance.
    """
    def __init__(self, mixup_alpha=0.8, cutmix_alpha=1.0, num_classes=100):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.num_classes = num_classes

    def __call__(self, batch):
        images, labels = zip(*batch)
        images = torch.stack(images, dim=0)  # (B, 3, H, W)
        labels_tensor = torch.tensor(labels, dtype=torch.long)
        labels_oh = F.one_hot(labels_tensor, num_classes=self.num_classes).float()

        bsz = images.size(0)

        do_mixup = (random.random() < 0.5)
        if do_mixup and (self.mixup_alpha > 0.0):
            alpha = self.mixup_alpha
            lam = np.random.beta(alpha, alpha)
            index = torch.randperm(bsz)
            images = lam*images + (1-lam)*images[index,:]
            labels_oh = lam*labels_oh + (1-lam)*labels_oh[index,:]
        else:
            alpha = self.cutmix_alpha
            if alpha > 0.0:
                lam = np.random.beta(alpha, alpha)
                _,_,H,W = images.shape
                cut_rat = np.sqrt(1.0 - lam)
                cutW = int(W*cut_rat)
                cutH = int(H*cut_rat)
                cx = np.random.randint(W)
                cy = np.random.randint(H)
                x1 = np.clip(cx - cutW//2, 0, W)
                x2 = np.clip(cx + cutW//2, 0, W)
                y1 = np.clip(cy - cutH//2, 0, H)
                y2 = np.clip(cy + cutH//2, 0, H)
                index = torch.randperm(bsz)
                images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]
                lam = 1 - ((x2 - x1)*(y2 - y1)/(W*H))
                labels_oh = lam*labels_oh + (1-lam)*labels_oh[index,:]

        return images, labels_oh

In [8]:
global_step = 0

In [ ]:
def train_one_epoch(model, criterion, dataloader, optimizer, device, epoch,
                    loss_scaler, max_norm=0, accum_iter=1):
    model.train()
    total_steps = len(dataloader)
    running_loss = 0.0

    optimizer.zero_grad()
    for step, (images, targets) in enumerate(tqdm(dataloader, total=total_steps, desc=f"Train Epoch {epoch}")):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)

        loss_value = loss.item()
        running_loss += loss_value

        loss = loss / accum_iter
        loss_scaler.scale(loss).backward()

        global global_step
        global_step += 1 

        wandb.log({
            "train_loss": loss_value,
            "lr": optimizer.param_groups[0]["lr"],
        }, step=global_step)


        if (step+1) % accum_iter == 0:
            loss_scaler.step(optimizer)
            loss_scaler.update()
            optimizer.zero_grad()

    avg_loss = running_loss / total_steps
    print(f"Train Epoch [{epoch}]   Loss: {avg_loss:.4f}")
    return avg_loss

@torch.no_grad()
def evaluate(model, criterion, dataloader, device, epoch):
    model.eval()
    total_loss = 0.0
    correct_top1 = 0
    total_samples = 0
    for images, targets in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        # Convert targets to one-hot encoding, as we aren't cutmixing, should be correct(?????)
        targets = F.one_hot(targets, num_classes=500).float()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)

        pred_class = outputs.argmax(dim=1)
        true_class = targets.argmax(dim=1)
        correct_top1 += (pred_class == true_class).sum().item()
        total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    acc1 = 100.0 * correct_top1 / total_samples
    print(f"Val Epoch [{epoch}]   Loss: {avg_loss:.4f}   Top-1: {acc1:.2f}%")
    return acc1, avg_loss


In [ ]:
epochs = 300
num_classes = 500
image_size = 224
train_batch_size = 1024      
accum_iter = 2                
effective_batch_size = train_batch_size * accum_iter
assert effective_batch_size == 2048


lr = 3e-3
weight_decay = 0.02
warmup_epochs = 5
drop_path=0
color_jitter=0.3
mixup_alpha=0.8
cutmix_alpha=1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Effective batch size = {effective_batch_size}, LR = {lr}")

Using device: cuda
Effective batch size = 2048, LR = 0.003


In [ ]:
train_transform = three_augment_transform(input_size=image_size, color_jitter=color_jitter)
val_transform = eval_transform(input_size=image_size)

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=val_dir, transform=val_transform)

val_dataset.samples = [s for s in val_dataset.samples if s[1] < 500]  # We only really need this, but whatever
val_dataset.targets = [s[1] for s in val_dataset.samples]
val_dataset.classes = val_dataset.classes[:500]
val_dataset.class_to_idx = {cls: idx for idx, cls in enumerate(val_dataset.classes)}


train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    drop_last=True,
    collate_fn=MixupCutmixCollator(mixup_alpha=mixup_alpha, cutmix_alpha=cutmix_alpha, num_classes=num_classes),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)

# 5) Model
model = DeiT_Tiny_LS(
    img_size=image_size,
    patch_size=16,
    num_classes=num_classes,
    drop_path_rate=drop_path,
    use_cls_token=True,
)
model.to(device)

criterion = nn.BCEWithLogitsLoss().to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay
)

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return float(epoch + 1) / warmup_epochs
    t = (epoch - warmup_epochs + 1) / (epochs - warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * t))


lr_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

loss_scaler = torch.GradScaler('cuda')

best_acc1 = 0.0

In [14]:
wandb.init(
    project="deit_tiny_training",
    name="multicls_500",
    config={
        "epochs": epochs,
        "batch_size": train_batch_size,
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "warmup_epochs": warmup_epochs,
        "drop_path": drop_path,
        "color_jitter": color_jitter,
        "mixup_alpha": mixup_alpha,
        "cutmix_alpha": cutmix_alpha,
    },
)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jchencxh to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
for epoch in range(1, epochs+1):
    train_loss = train_one_epoch(
        model, criterion, train_loader, optimizer, device, epoch,
        loss_scaler, accum_iter=accum_iter
    )

    acc1, val_loss = evaluate(model, criterion, val_loader, device, epoch)
    if acc1 > best_acc1:
        best_acc1 = acc1
        torch.save(model.state_dict(), "best_deit_tiny.pth")

    lr_scheduler.step()


    wandb.log({
        "epoch": epoch,
        "val_loss": val_loss,
        "accuracy": acc1,
    }, step=global_step)

    print(f"Epoch {epoch} done. Best Acc={best_acc1:.2f}%\n")


print(f"Best top-1 accuracy: {best_acc1:.2f}%")

Train Epoch 1: 100%|██████████| 306/306 [07:49<00:00,  1.54s/it]

Train Epoch [1]   Loss: 0.0344


Val Epoch [1]   Loss: 0.0144   Top-1: 0.17%
Epoch 1 done. Best Acc=0.17%



Train Epoch 2: 100%|██████████| 306/306 [08:16<00:00,  1.62s/it]

Train Epoch [2]   Loss: 0.0144


...

Val Epoch [63]   Loss: 0.0073   Top-1: 39.82%
Epoch 63 done. Best Acc=39.82%



Train Epoch 64: 100%|██████████| 306/306 [08:19<00:00,  1.63s/it]

Train Epoch [64]   Loss: 0.0113


Val Epoch [64]   Loss: 0.0075   Top-1: 38.97%
Epoch 64 done. Best Acc=39.82%



Train Epoch 65:  91%|█████████ | 278/306 [07:40<00:58,  2.08s/it]

In [ ]:
wandb.finish()

In [ ]:
from google.colab import runtime
runtime.unassign()